In [1]:
import pandas as pd
import numpy as np

In [15]:
df = pd.read_csv('winequality-white.csv',sep=';')

In [16]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [17]:
df.info

<bound method DataFrame.info of       fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0               7.0              0.27         0.36            20.7      0.045   
1               6.3              0.30         0.34             1.6      0.049   
2               8.1              0.28         0.40             6.9      0.050   
3               7.2              0.23         0.32             8.5      0.058   
4               7.2              0.23         0.32             8.5      0.058   
...             ...               ...          ...             ...        ...   
4893            6.2              0.21         0.29             1.6      0.039   
4894            6.6              0.32         0.36             8.0      0.047   
4895            6.5              0.24         0.19             1.2      0.041   
4896            5.5              0.29         0.30             1.1      0.022   
4897            6.0              0.21         0.38             0.8      0.020

In [11]:
df.columns = df.columns.str.strip()

In [13]:
print(df.isnull().sum())

fixed acidity;"volatile acidity";"citric acid";"residual sugar";"chlorides";"free sulfur dioxide";"total sulfur dioxide";"density";"pH";"sulphates";"alcohol";"quality"    0
dtype: int64


In [18]:
df['quality'] = df['quality'].apply(lambda x: 1 if x >= 7 else 0)

In [19]:
print(df['quality'].value_counts())

quality
0    3838
1    1060
Name: count, dtype: int64


In [21]:
X = df.drop('quality',axis=1)
y = df['quality']

In [24]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [28]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

/Users/danishasghar/Desktop/LSTM RNN/tf_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [31]:
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier()
tree.fit(X_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [34]:
y_pred = tree.predict(X_test)

In [39]:
from sklearn.metrics import accuracy_score,classification_report
score = accuracy_score(y_test,y_pred)
score

0.2316326530612245

In [40]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       753
           1       0.23      1.00      0.38       227

    accuracy                           0.23       980
   macro avg       0.12      0.50      0.19       980
weighted avg       0.05      0.23      0.09       980



/Users/danishasghar/Desktop/LSTM RNN/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danishasghar/Desktop/LSTM RNN/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danishasghar/Desktop/LSTM RNN/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [52]:
from sklearn.model_selection import GridSearchCV
params = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

In [55]:
grid = GridSearchCV(estimator=tree, param_grid=params,scoring='accuracy',cv=5)
grid.fit(X_train,y_train)

,estimator,DecisionTreeClassifier()
,param_grid,"{'max_depth': [3, 5, ...], 'min_samples_split': [2, 5, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [91]:
dt_acc = grid.best_score_
dt_acc

0.8677838376730003

In [57]:
grid.best_params_

{'max_depth': None, 'min_samples_split': 2}

In [60]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier()
forest.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [61]:
y_pred = forest.predict(X_test)

In [65]:
score = accuracy_score(y_test,y_pred)
score

0.7683673469387755

In [66]:
params = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None]
}

In [70]:
grid = GridSearchCV(estimator = forest, param_grid =params , cv = 5, scoring = 'accuracy')
grid.fit(X_train,y_train)

,estimator,RandomForestClassifier()
,param_grid,"{'max_depth': [5, 10, ...], 'n_estimators': [50, 100]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [94]:
rf_acc = grid.best_score_
rf_acc

0.8677838376730003

In [77]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(X_train,y_train)


,priors,None
,var_smoothing,1e-09


In [78]:
y_pred = nb.predict(X_test)

In [80]:
score = accuracy_score(y_test,y_pred)
score

0.7683673469387755

In [86]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential

In [88]:
model = Sequential([
    keras.layers.Dense(32,activation='relu',input_shape=(X_train.shape[1],)),
    keras.layers.Dense(16,activation='relu'),
    keras.layers.Dense(1,activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)
model.fit(X_train,y_train,epochs=30,batch_size=16, validation_split=0.2,
    verbose=0)

nn_loss, nn_acc = model.evaluate(X_test, y_test)

print("Neural Network Accuracy:", nn_acc)

2026-03-24 07:12:11.254478: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-03-24 07:12:11.254554: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-24 07:12:11.254564: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-24 07:12:11.254659: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-24 07:12:11.254727: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-03-24 07:12:12.052630: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


31/31 [==============================] - 0s 7ms/step - loss: 4.2373 - accuracy: 0.7602
Neural Network Accuracy: 0.7602040767669678


In [96]:
print("\n Model Comaprison:")
print(f"Decision Tree: {dt_acc:.4f}")
print(f"Random Forest: {rf_acc:.4f}")
print(f"Naive bayes: {score:.4f}")
print(f"Neural network: {nn_acc:.4f}")




 Model Comaprison:
Decision Tree: 0.8678
Random Forest: 0.8678
Naive bayes: 0.7684
Neural network: 0.7602


In [1]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
